# Item 92: Profile Before Optimising

## Notes

-   Python’s dynamic behaviour means that sometimes the performance cost
    of operations is not intuitive
    -   Some operations that would seem slow are fast, e.g.
        1.  String manipulation
        2.  Use of Generators
    -   Some basic operations that would seem fast are slow, e.g.
        1.  Attribute access
        2.  Function calls
-   Best way to determine performance and identify bottlenecks is to
    actually profile
    -   Python provides a built-in *profiler*
-   Profiling lets you focus on the real bottlenecks
-   For example, a classic profiling scenario is a sort
    -   Insertion sort works by inserting the next item in the unsorted
        part of a list into it’s correct position in the sorted part of
        the list
    -   We’ll demo by implementing an inefficient `insert_value`
        function to determine the insertion point via a linear scan

In [1]:
def insert_value(array, value):
    for i, existing in enumerate(array):
        if existing > value:
            array.insert(i, value)
            return
    array.append(value)


def insertion_sort(data):
    result = []
    for value in data:
        insert_value(result, value)
    return result

-   We can then profile the code
    -   Generate a list of random numbers and define a `test` function
        (See [Item 39](../../Chapter_05/Item_039/item_039.qmd))
-   Python provides two profilers
    -   The pure python `profile`
    -   C extension module `cProfile`
    -   Prefer `cProfile` because it has less of a performance overhead
        -   `profile` has an overhead that can skew the results
-   We then instantiate a `Profile` object
    -   Can then run a profile on a function via the `runcall` method
-   To extract the statistics we can use the `pstats` built-in
    -   Then use the `Stats` class
        -   Provides methods to adjust how we display the profiling

In [2]:
from cProfile import Profile
from pstats import Stats
from random import randint

max_size = 12**4
data = [randint(0, max_size) for _ in range(max_size)]
test = lambda: insertion_sort(data)


def insert_value(array, value):
    for i, existing in enumerate(array):
        if existing > value:
            array.insert(i, value)
            return
    array.append(value)


def insertion_sort(data):
    result = []
    for value in data:
        insert_value(result, value)
    return result


# Run the profile
profiler = Profile()
profiler.runcall(test)

stats = Stats(profiler)
stats.strip_dirs()
stats.sort_stats("cumulative")
stats.print_stats()

         41888 function calls (41884 primitive calls) in 3.554 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    20736    3.525    0.000    3.548    0.000 2565863959.py:10(insert_value)
        4    0.000    0.000    3.542    0.886 base_events.py:1977(_run_once)
        1    0.001    0.001    3.390    3.390 2565863959.py:7(<lambda>)
        1    0.003    0.003    2.884    2.884 2565863959.py:18(insertion_sort)
        3    0.001    0.000    0.152    0.051 selectors.py:435(select)
    20726    0.023    0.000    0.023    0.000 {method 'insert' of 'list' objects}
        1    0.000    0.000    0.005    0.005 iostream.py:348(<lambda>)
        1    0.000    0.000    0.005    0.005 iostream.py:350(_really_send)
        1    0.000    0.000    0.005    0.005 socket.py:700(send_multipart)
        2    0.000    0.000    0.000    0.000 {built-in method time.sleep}
        3    0.000    0.000    0.000    0.000 events.py:92(_run)
  

-   The profiler shows a range of statistics, namely
    1.  **ncalls:** The number of times the function is called
    2.  **tottime:** Number of seconds spent executing the function,
        excluding executing sub-functions
    3.  **tottime percall:** Average number of time spent in a function
        each time it is called (excluding sub-functions it calls)
    4.  **cumtime:** Cumulative number of seconds spent executing this
        function, including in sub-functions
    5.  **cumtime percall:** Average number of seconds spent executing
        this function each call, including in sub-functions
-   As expected the biggest consumer is our `insert_value` function
    -   We can reimplement this with a *binary search*
    -   Provided by the `bisect` built-in module

> **Note**
>
> When profiling a program be sure to measure the actual code and not
> any external systems. Functions that access disk or networks will tend
> to otherwise dominate the profile because these operations are orders
> of magnitude slower than the actual CPU. If your program or system
> provides a cache then improperly warming it may cause subsequent tests
> to give very different results

In [3]:
from bisect import bisect_left
from cProfile import Profile
from pstats import Stats
from random import randint

max_size = 12**4
data = [randint(0, max_size) for _ in range(max_size)]
test = lambda: insertion_sort(data)


def insert_value(array, value):
    i = bisect_left(array, value)
    array.insert(i, value)


def insertion_sort(data):
    result = []
    for value in data:
        insert_value(result, value)
    return result


# Run the profile
profiler = Profile()
profiler.runcall(test)

stats = Stats(profiler)
stats.strip_dirs()
stats.sort_stats("cumulative")
stats.print_stats()

         62531 function calls (62527 primitive calls) in 0.041 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    20736    0.008    0.000    0.037    0.000 2000139989.py:11(insert_value)
        1    0.000    0.000    0.035    0.035 2000139989.py:8(<lambda>)
        1    0.003    0.003    0.035    0.035 2000139989.py:16(insertion_sort)
    20736    0.022    0.000    0.022    0.000 {method 'insert' of 'list' objects}
    20736    0.007    0.000    0.007    0.000 {built-in method _bisect.bisect_left}
        1    0.000    0.000    0.005    0.005 iostream.py:348(<lambda>)
        1    0.000    0.000    0.005    0.005 iostream.py:350(_really_send)
        1    0.001    0.001    0.005    0.005 socket.py:700(send_multipart)
        1    0.000    0.000    0.000    0.000 base_events.py:1977(_run_once)
        1    0.000    0.000    0.000    0.000 events.py:92(_run)
        1    0.000    0.000    0.000    0.000 {method 'run' of '

-   The new implementation runs much faster
    -   The function with the highest **tottime** is now the `insert`
        method on `list`
-   One scenario that occurs frequently is a common utility function
    dominating the execution time
    -   Can be difficult to interpret these scenarios because the
        default profiler display will mix all the different calls
        together
        -   Would be better to isolate by call site
-   Consider the following example,

In [4]:
from cProfile import Profile
from pstats import Stats


def utility(a, b):
    c = 1
    for i in range(100):
        c += a * b


def first_function():
    for _ in range(1000):
        utility(4, 5)


def second_function():
    for _ in range(10):
        utility(1, 3)


def program():
    for _ in range(20):
        first_function()
        second_function()


profiler = Profile()
profiler.runcall(program)

stats = Stats(profiler)
stats.strip_dirs()
stats.sort_stats("cumulative")
stats.print_stats()

         20750 function calls (20745 primitive calls) in 0.092 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    20200    0.073    0.000    0.073    0.000 3007661153.py:5(utility)
       20    0.004    0.000    0.072    0.004 3007661153.py:11(first_function)
      2/1    0.014    0.007    0.064    0.064 3007661153.py:21(program)
        2    0.000    0.000    0.011    0.005 events.py:92(_run)
        2    0.000    0.000    0.011    0.005 {method 'run' of '_contextvars.Context' objects}
        2    0.000    0.000    0.011    0.005 zmqstream.py:573(_handle_events)
        1    0.000    0.000    0.011    0.011 asyncio.py:206(_handle_events)
        2    0.000    0.000    0.010    0.005 zmqstream.py:614(_handle_recv)
        2    0.000    0.000    0.010    0.005 zmqstream.py:546(_run_callback)
        2    0.000    0.000    0.010    0.005 iostream.py:229(_handle_event)
        2    0.000    0.000    0.010    0.005 iostream

-   We can see that `utility` is called the most
    -   But not obvious why, or which caller is most responsible
-   We can use `print_callers` on the `Stats` object
    -   Show’s which caller contributed to each function’s profiling
        information

In [5]:
from cProfile import Profile
from pstats import Stats


def utility(a, b):
    c = 1
    for i in range(100):
        c += a * b


def first_function():
    for _ in range(1000):
        utility(4, 5)


def second_function():
    for _ in range(10):
        utility(1, 3)


def program():
    for _ in range(20):
        first_function()
        second_function()


profiler = Profile()
profiler.runcall(program)

stats = Stats(profiler)
stats.strip_dirs()
stats.sort_stats("cumulative")
stats.print_callers()

   Ordered by: cumulative time

Function                                              was called by...
                                                          ncalls  tottime  cumtime
base_events.py:1977(_run_once)                        <-       0    0.000    0.000  base_events.py:1977(_run_once)
3101860672.py:5(utility)                              <- 18200/791    0.066    0.003  3101860672.py:11(first_function)
                                                             200    0.001    0.001  3101860672.py:16(second_function)
                                                             561    0.002    0.002  selectors.py:435(select)
                                                         855/208    0.003    0.001  socket.py:700(send_multipart)
                                                             382    0.001    0.001  {method 'poll' of 'select.epoll' objects}
3101860672.py:11(first_function)                      <-       0    0.000    0.000  3101860672.py:21(program)
   

-   Functions called are listed on the left
    -   Functions that call that function are listed on the right
-   This lets us see that `first_function` is clearly the main culprit
-   An alternative is the `print_calles` method
    -   Provides a top down view of which functions another function
        calls

In [6]:
from cProfile import Profile
from pstats import Stats


def utility(a, b):
    c = 1
    for i in range(100):
        c += a * b


def first_function():
    for _ in range(1000):
        utility(4, 5)


def second_function():
    for _ in range(10):
        utility(1, 3)


def program():
    for _ in range(20):
        first_function()
        second_function()


profiler = Profile()
profiler.runcall(program)

stats = Stats(profiler)
stats.strip_dirs()
stats.sort_stats("cumulative")
stats.print_callees()

   Ordered by: cumulative time

Function                                              called...
                                                          ncalls  tottime  cumtime
base_events.py:1977(_run_once)                        ->      17    0.003    0.063  3103966480.py:11(first_function)
                                                              18    0.000    0.001  3103966480.py:16(second_function)
                                                               4    0.000    0.000  base_events.py:766(time)
                                                               2    0.000    0.009  base_events.py:1977(_run_once)
                                                               2    0.000    0.000  events.py:92(_run)
                                                               2    0.000    0.000  selector_events.py:744(_process_events)
                                                             2/0    0.000    0.000  selectors.py:435(select)
                          

-   There are further tools for analysing performance once a basic
    profiling has been conducted (See [Item
    93](../Item_093/item_093.qmd) and [Item
    98](../Item_098/item_098.qmd))
-   There is also a broader community of profiling tools, e.g.
    -   Line profilers
    -   Sampling profilers
    -   Linux’s `perf` tool
    -   Memory usage profilers

## Things to Remember

-   Before attempting to optimise always profile to identify bottlenecks
-   Prefer the `cProfile` profiler over `profile` for less profiling
    overhead
-   Use `runcall` on a `Profile` object to profile a tree of function
    calls isolated from a larger program
-   Use `Stats` from the `pstats` module to select and print statistics
    generated by a profiler